# FMA (`x[i] * y[i] + z[i]`) — GPU vs CPU vs CPU-vectorized

All three versions use `x[i] = y[i] = z[i] = i`, so `R[i] = i*i + i`.

- **GPU**: CUDA kernel, one thread per element
- **CPU (plain)**: scalar loop, compiled with `-O0`
- **CPU (vectorized)**: same loop, compiled with `-O3` so the compiler auto-vectorizes (SSE/AVX)

Per-element / per-thread `printf`s are kept in the source but commented out. Each binary prints its own elapsed time.

In [6]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2026 NVIDIA Corporation
Built on Thu_Mar_19_22:28:55_Pacific_Daylight_Time_2026
Cuda compilation tools, release 13.2, V13.2.78
Build cuda_13.2.r13.2/compiler.37668154_0
Wed Apr 29 14:41:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.59                 Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A 

## 1. GPU

In [36]:
%%writefile fma_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N       262144
#define ITERS   1024
#define THREADS 256
#define BLOCKS  (N / THREADS)

__global__ void fmaKernel(const float *X, const float *Y, const float *Z, float *R, int n) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    // int warp_id = threadIdx.x / 32;
    // int lane_id = threadIdx.x % 32;
    // int smid;
    // asm("mov.u32 %0, %%smid;" : "=r"(smid));

    if (gid < n) {
        R[gid] = X[gid] * Y[gid] + Z[gid];
    }

    // if (lane_id == 0 && gid < n && blockIdx.x < 4) {
    //     printf("Block %d | SM %d | Warp %d | Global Thread %d\n",
    //            blockIdx.x, smid, warp_id, gid);
    // }
    // if (gid < n) {
    //     printf("R[%d] = %f\n", gid, R[gid]);
    // }
}

int main() {
    size_t bytes = N * sizeof(float);

    float *hX = (float*)malloc(bytes);
    float *hY = (float*)malloc(bytes);
    float *hZ = (float*)malloc(bytes);
    float *hR = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        hX[i] = (float)i;
        hY[i] = (float)i;
        hZ[i] = (float)i;
    }

    float *dX, *dY, *dZ, *dR;
    cudaMalloc(&dX, bytes);
    cudaMalloc(&dY, bytes);
    cudaMalloc(&dZ, bytes);
    cudaMalloc(&dR, bytes);

    cudaMemcpy(dX, hX, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dY, hY, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dZ, hZ, bytes, cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // warm-up
    fmaKernel<<<BLOCKS, THREADS>>>(dX, dY, dZ, dR, N);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    for (int it = 0; it < ITERS; it++) {
        fmaKernel<<<BLOCKS, THREADS>>>(dX, dY, dZ, dR, N);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_ms = 0.0f;
    cudaEventElapsedTime(&gpu_ms, start, stop);

    cudaMemcpy(hR, dR, bytes, cudaMemcpyDeviceToHost);

    // for (int i = 0; i < N; i++) {
    //     printf("R[%d] = %f\n", i, hR[i]);
    // }

    printf("=== GPU ===\n");
    printf("N = %d, ITERS = %d, BLOCKS = %d, THREADS = %d\n", N, ITERS, BLOCKS, THREADS);
    printf("GPU total: %.4f ms  (%.4f us/iter)\n", gpu_ms, (gpu_ms * 1000.0) / ITERS);
    printf("Sample: R[0]=%.2f  R[1]=%.2f  R[100]=%.2f  R[%d]=%.2f\n",
           hR[0], hR[1], hR[100], N-1, hR[N-1]);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(dX); cudaFree(dY); cudaFree(dZ); cudaFree(dR);
    free(hX); free(hY); free(hZ); free(hR);
    return 0;
}


Overwriting fma_cuda.cu


In [37]:
!nvcc -O3 -arch=sm_86 fma_cuda.cu -o fma_cuda
!fma_cuda.exe

fma_cuda.cu
tmpxft_0000152c_00000000-7_fma_cuda.cudafe1.cpp
=== GPU ===
N = 262144, ITERS = 1024, BLOCKS = 1024, THREADS = 256
GPU total: 17.8964 ms  (17.4770 us/iter)
Sample: R[0]=0.00  R[1]=2.00  R[100]=10100.00  R[262143]=68719214592.00


## 2. CPU

Single source file, compiled twice — once with `-O0` (scalar) and once with `-O3` (auto-vectorized).

In [33]:
%%writefile fma_cpu.cpp
#include <stdio.h>
#include <stdlib.h>
#include <chrono>

#define N     262144
#define ITERS 1024

#if defined(_MSC_VER)
  #define RESTRICT __restrict
#else
  #define RESTRICT __restrict__
#endif

void cpu_fma(const float * RESTRICT X,
             const float * RESTRICT Y,
             const float * RESTRICT Z,
             float       * RESTRICT R, int n) {
    for (int i = 0; i < n; i++) {
        R[i] = X[i] * Y[i] + Z[i];
    }
}

int main() {
    size_t bytes = N * sizeof(float);

    float *X = (float*)malloc(bytes);
    float *Y = (float*)malloc(bytes);
    float *Z = (float*)malloc(bytes);
    float *R = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        X[i] = (float)i;
        Y[i] = (float)i;
        Z[i] = (float)i;
    }

    // warm-up
    cpu_fma(X, Y, Z, R, N);

    auto t0 = std::chrono::high_resolution_clock::now();
    for (int it = 0; it < ITERS; it++) {
        cpu_fma(X, Y, Z, R, N);
    }
    auto t1 = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    // for (int i = 0; i < N; i++) {
    //     printf("R[%d] = %f\n", i, R[i]);
    // }

    printf("=== CPU ===\n");
    printf("N = %d, ITERS = %d\n", N, ITERS);
    printf("CPU total: %.4f ms  (%.4f us/iter)\n", cpu_ms, (cpu_ms * 1000.0) / ITERS);
    printf("Sample: R[0]=%.2f  R[1]=%.2f  R[100]=%.2f  R[%d]=%.2f\n",
           R[0], R[1], R[100], N-1, R[N-1]);

    free(X); free(Y); free(Z); free(R);
    return 0;
}


Overwriting fma_cpu.cpp


### 2a. CPU plain (`-O0`, scalar)

In [34]:
!nvcc fma_cpu.cpp -o fma_cpu_plain
!fma_cpu_plain.exe

fma_cpu.cpp
=== CPU ===
N = 262144, ITERS = 1024
CPU total: 520.7788 ms  (508.5730 us/iter)
Sample: R[0]=0.00  R[1]=2.00  R[100]=10100.00  R[262143]=68719214592.00


### 2b. CPU vectorized (`-O3`, auto-vectorized)

In [35]:
!nvcc -O3 fma_cpu.cpp -o fma_cpu_vec
!fma_cpu_vec.exe

fma_cpu.cpp
=== CPU ===
N = 262144, ITERS = 1024
CPU total: 82.5670 ms  (80.6318 us/iter)
Sample: R[0]=0.00  R[1]=2.00  R[100]=10100.00  R[262143]=68719214592.00
